In [29]:
from functools import lru_cache
from config.settings import get_settings
import numpy as np
import requests

settings = get_settings()
logger = settings.get_logger()

### FUNCTION TO FETCH LAST 100 DAILY CLOSE PRICES ON A TICKER

In [6]:
@lru_cache(maxsize=128)
def _fetch_daily_closes(ticker: str) -> tuple[tuple[str, float], ...] | str:
    """Returns list of closing prices (newest first) or an error string."""
    params = {
        "function": "TIME_SERIES_DAILY",
        "symbol": ticker,
        "outputsize": "compact",  # 100 trading days — sufficient for annualised vol
        "apikey": settings.vantage_alpha_api_key,
    }
    try:
        resp = requests.get(settings.vantage_alpha_base_url, params=params, timeout=10)
        resp.raise_for_status()
    except requests.RequestException as e:
        return str(e)

    data = resp.json()
    logger.info(f"Fetched data for {ticker}")

    if "Error Message" in data:
        return data["Error Message"]
    if "Information" in data:
        return data["Information"]  # rate-limit message

    time_series = data.get("Time Series (Daily)", {})
    if not time_series:
        return f"No time-series data returned for {ticker}"

    sorted_dates = sorted(time_series.keys(), reverse=True)
    closes = tuple((d, float(time_series[d]["4. close"])) for d in sorted_dates)
    logger.info(f"Extracted {len(closes)} closing prices for {ticker}")

    return closes

In [7]:
time_series_aapl = _fetch_daily_closes("AAPL")

In [12]:
time_series_msft = _fetch_daily_closes("MSFT")

In [14]:
print(time_series_aapl[:10])  # Display the first 10 entries
print(time_series_msft[:10])  # Display the first 10 entries

(('2026-06-16', 299.24), ('2026-06-15', 296.42), ('2026-06-12', 291.13), ('2026-06-11', 295.63), ('2026-06-10', 291.58), ('2026-06-09', 290.55), ('2026-06-08', 301.54), ('2026-06-05', 307.34), ('2026-06-04', 311.23), ('2026-06-03', 310.26))
(('2026-06-16', 393.83), ('2026-06-15', 399.76), ('2026-06-12', 390.74), ('2026-06-11', 390.34), ('2026-06-10', 397.36), ('2026-06-09', 403.41), ('2026-06-08', 411.74), ('2026-06-05', 416.67), ('2026-06-04', 428.05), ('2026-06-03', 427.34))


In [27]:
def align_time_series(tickers: list[str]) -> dict[str, list[float]]:
    """Aligns time series data for multiple tickers by date."""
    closes_by_ticker: dict[str, dict[str, float]] = {}

    for ticker in tickers:
        data = _fetch_daily_closes(ticker)
        if isinstance(data, str):
            raise ValueError(f"{ticker}: {data}")
        closes_by_ticker[ticker] = dict(data)
    
    # set(data) -> set of keys from date to close dict, which is set of dates
    # * unpacks this for each ticker in closes_by_ticker so inside intersectio is multuple set arguments
    # intersection ensures no None or empty values
    common_dates = sorted(set.intersection(*(set(data) for data in closes_by_ticker.values())))
    if len(common_dates) < 2:
        raise ValueError(f"Need >2 common dates accros all tickers, got {len(common_dates)}")
    
    logger.info("time series data aligned successfully")
    return {
        ticker: [closes_by_ticker[ticker][date] for date in common_dates]
        for ticker in tickers
    }

In [28]:
aligned_time_series = align_time_series(["AAPL", "MSFT"])

In [46]:
for ticker in aligned_time_series:
    print(f"{ticker}: {aligned_time_series[ticker]}\n")

AAPL: [248.04, 255.41, 258.27, 256.44, 258.28, 259.48, 270.01, 269.48, 276.49, 275.91, 278.12, 274.62, 273.68, 275.5, 261.73, 255.78, 263.88, 264.35, 260.58, 264.58, 266.18, 272.14, 274.23, 272.95, 264.18, 264.72, 263.75, 262.52, 260.29, 257.46, 259.88, 260.83, 260.81, 255.76, 250.12, 252.82, 254.23, 249.94, 248.96, 247.99, 251.49, 251.64, 252.62, 252.89, 248.8, 246.63, 253.79, 255.63, 255.92, 258.86, 253.5, 258.9, 260.49, 260.48, 259.2, 258.83, 266.43, 263.4, 270.23, 273.05, 266.17, 273.17, 273.43, 271.06, 267.61, 270.71, 270.17, 271.35, 280.14, 276.83, 284.18, 287.51, 287.44, 293.32, 292.68, 294.8, 298.87, 298.21, 300.23, 297.84, 298.97, 302.25, 304.99, 308.82, 308.33, 310.85, 312.51, 312.06, 306.31, 315.2, 310.26, 311.23, 307.34, 301.54, 290.55, 291.58, 295.63, 291.13, 296.42, 299.24]

MSFT: [465.95, 470.28, 480.58, 481.63, 433.5, 430.29, 423.37, 411.21, 414.19, 393.67, 401.14, 413.6, 413.27, 404.37, 401.84, 401.32, 396.86, 399.6, 398.46, 397.23, 384.47, 389.0, 400.6, 401.72, 392.74

In [58]:
def _log_returns_matrix(aligned_data: dict[str, list[float]], tickers: list[str]) -> np.ndarray:
    """Calculates the returns matrix from aligned closing prices."""
    prices = np.array([aligned_data[ticker] for ticker in tickers]).T
    # need to transpose because we want each column to represent a ticker and each row to represent a date
    returns = np.diff(np.log(prices), axis=0)  # log returns
    logger.info("returns matrix calculated successfully")
    return returns

In [56]:
returns_matrix = _returns_matrix(aligned_time_series, ["AAPL", "MSFT"])

In [57]:
returns_matrix.shape

(99, 2)